
# 06 — Zero-shot foundation models

The question this dataset's main weakness makes unavoidable:

> **With only ~900 samples per site, is bespoke per-operator training worth it, or does
> a foundation model that has never seen this network do just as well?**

Chronos-Bolt is evaluated strictly zero-shot — no fitting, no fine-tuning, not even a
scaling constant — on exactly the fold schedule, lead times and test blocks that
notebook 04 uses, so the numbers drop into the same table.

Two things make this more than a routine extra baseline.

**It is quantile-native.** Chronos-Bolt emits quantiles directly, so it plugs into the
allocation layer with no quantile-regression step. We can therefore ask whether its
*uncalibrated* quantiles deliver their nominal service level.

**It assumes regular sampling, which these traces violate.** A foundation model
consumes a sequence of values with no timestamps. It cannot know that a step is 86
minutes on one trace and 99 on the other. Everything notebook 00 corrects by *measuring*
the sampling rate, this model class structurally cannot see.

Runs on CPU — no GPU needed. `pip install chronos-forecasting`, then
`python experiments/run_foundation.py` (~7 min).

In [1]:
# --- Bootstrap: works locally and on Colab ---------------------------------
# Locally this just finds the repository root. On Colab the repo is not on the VM
# yet, so it is cloned first. The repository is PRIVATE, which means the clone
# needs a GitHub token -- put one in Colab Secrets (the key icon in the left
# sidebar) under the name GH_TOKEN and enable notebook access. See docs/COLAB.md.
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
REPO = "github.com/sad-code-at/bwalloc.git"

ROOT = Path.cwd()
while not (ROOT / "src" / "bwalloc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "src" / "bwalloc").exists():
    target = Path("/content/bwalloc")
    if not (target / "src" / "bwalloc").exists():
        try:
            from google.colab import userdata
            token = userdata.get("GH_TOKEN")
        except Exception:
            token = None
        if not token:
            raise SystemExit(
                "Could not find the repository, and no GH_TOKEN is available. "
                "On Colab: add a GitHub token in Secrets (the key icon) as "
                "GH_TOKEN, enable notebook access for this notebook, and re-run "
                "-- see docs/COLAB.md. Locally: run this notebook from inside "
                "the repository."
            )
        # The token never reaches stdout: git is quiet and errors are sanitised.
        rc = os.system(f"git clone -q https://{token}@{REPO} {target} 2>/dev/null")
        if rc != 0 or not (target / "src" / "bwalloc").exists():
            raise SystemExit(
                "git clone failed. Check that GH_TOKEN is valid, not expired, and "
                "has read access to this repository (Contents: Read)."
            )
    os.chdir(target)
    ROOT = target

sys.path.insert(0, str(ROOT / "src"))

try:
    import xgboost  # noqa: F401
except ImportError:
    !pip install -q xgboost

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import bwalloc as bw
from bwalloc.plots import use_paper_style

bw.set_seed()
use_paper_style()
pd.set_option("display.width", 200)
RESULTS = ROOT / "experiments" / "results"
# Scratch output for the exploratory notebooks. Only 07_paper_figures writes into
# paper/figures -- otherwise running notebook 00 or 05 silently overwrites a figure
# the paper cites, which is exactly the kind of drift this project exists to remove.
FIGURES = ROOT / "notebooks" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("bwalloc", bw.__version__, "| results:", RESULTS)

bwalloc 0.1.0 | results: D:\L4-T-1\EEE 402\project\bwalloc\experiments\results


## Zero-shot against models trained on this operator

In [2]:

accuracy = pd.read_csv(RESULTS / "foundation_accuracy.csv")
zero = (
    accuracy.groupby(["operator", "model", "lead_hours"])
    .agg(rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"))
    .reset_index()
)

for operator in ("gp", "robi"):
    trained = pd.read_csv(RESULTS / f"horizon_{operator}.csv")
    table = trained.pivot_table(index="model", columns="lead_hours", values="rmse_mean")
    z = zero[zero["operator"] == operator].pivot_table(
        index="model", columns="lead_hours", values="rmse_mean"
    )
    print(operator.upper())
    display(pd.concat([table, z]).round(2))

GP


lead_hours,1.43,2.87,5.73,11.47,24.37
model,,,,,
persistence_h,12.12,17.41,25.11,31.16,18.26
random_forest,10.40,11.85,12.48,12.83,12.82
ridge,11.26,14.45,15.87,14.50,15.08
seasonal_naive_17,18.31,18.31,18.35,18.38,18.26
xgboost,10.73,12.18,12.36,12.79,13.01
chronos-bolt-small,11.18,13.53,13.50,14.41,15.20


ROBI


lead_hours,1.65,3.30,6.60,11.55,24.75
model,,,,,
persistence_h,29.75,45.07,65.97,75.95,26.49
random_forest,20.77,25.23,27.27,28.01,22.69
ridge,22.72,28.14,30.91,30.33,23.30
seasonal_naive_15,26.26,26.26,26.27,26.24,26.49
xgboost,21.11,24.04,27.23,27.74,24.25
chronos-bolt-small,30.54,38.84,40.34,42.56,43.63


## Do its own quantiles deliver their nominal level?

In [3]:

calibration = pd.read_csv(RESULTS / "foundation_calibration.csv")
calibration["c"] = calibration["n"] * calibration["coverage"]
pooled = (
    calibration.groupby(["operator", "model", "tau", "method"])
    .agg(c=("c", "sum"), n=("n", "sum"),
         natively_expressible=("natively_expressible", "first"))
    .assign(achieved=lambda d: d["c"] / d["n"])
    .reset_index()
)
pooled["gap"] = pooled["achieved"] - pooled["tau"]
pooled.pivot_table(index=["operator", "tau", "natively_expressible"],
                   columns="method", values=["achieved", "gap"]).round(3)

achieved                 gap          
method                             conformal zero_shot conformal zero_shot
operator tau  natively_expressible                                        
gp       0.80 True                     0.815     0.791     0.015    -0.009
         0.90 True                     0.903     0.886     0.003    -0.014
         0.95 False                    0.941     0.886    -0.009    -0.064
robi     0.80 True                     0.813     0.662     0.013    -0.138
         0.90 True                     0.909     0.786     0.009    -0.114
         0.95 False                    0.950     0.786     0.000    -0.164


### The ceiling that matters for provisioning

Chronos-Bolt's quantile head was trained on levels **0.1 to 0.9 only**. A request for
τ = 0.95 is silently clipped to τ = 0.90 and returns the same numbers — which is why the
`natively_expressible` column is False there.

That is not a configuration detail, it is a limit on what the model can be used for.
Since τ\* = κ/(1+κ), a ceiling of 0.9 corresponds to a cost asymmetry of only **κ = 9**.
An operator for whom under-provisioning costs 20× more than over-provisioning needs
τ\* = 0.952, and cannot get it from the model's own quantile head at all.

So for this application, calibration on top of the foundation model is not an optional
refinement — it is what makes the model usable. The `conformal` rows show split-conformal
calibration on held-out residuals restoring the levels the model cannot express itself.